# 83 — Simulate10Next: Conqueror & Supplier Tests

Test notebook for `StrategyPipeline` from `82-Simulate10Next_Conqueror_Supplier.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

In [1]:
%run 82-Simulate10Next_Conqueror_Supplier.py

In [2]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
# import kaggle_environments as ke

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

# Testing new function

In [3]:
def _04_score_and_decide(attacks_with_angle: pd.DataFrame, player_id: int) -> list:
    if attacks_with_angle.empty:
        return []

    moves = []

    # Comet evasion
    awa_comets = attacks_with_angle[attacks_with_angle["nature_src"] == "comet"]
    if not awa_comets.empty:
        x_off = (awa_comets["x_src"] - GameConfig.CENTER).abs().max() or 0
        y_off = (awa_comets["y_src"] - GameConfig.CENTER).abs().max() or 0
        if max(x_off, y_off) > 45:
            moves += (
                awa_comets[awa_comets["ships_sent"] <= awa_comets["ships_min"]]
                .sort_values(["ships_sent", "step"], ascending=[False, True])
                .groupby("id_src", sort=False)
                .first()
                .reset_index()
                [["id_src", "final_angle", "ships_sent"]]
                .values.tolist()
            )
            id_to_avoid = awa_comets["id_src"].unique().tolist()
            attacks_with_angle = attacks_with_angle[~attacks_with_angle["id_src"].isin(id_to_avoid)]

    # Top-5 targets per source planet (cheapest by step then ships_sent)
    top5_ids = (
        attacks_with_angle
        .sort_values(["step", "ships_sent"])
        .groupby(["id_src", "id"], sort=False)
        .first()
        .reset_index()
        .sort_values(["step", "ships_sent"])
        .groupby("id_src", sort=False)
        .head(5)
        [["id_src", "id"]]
        .assign(is_top5=True)
    )

    # Source planet IDs owned by player (id_src is already filtered to player's planets)
    mine_src_ids = set(attacks_with_angle["id_src"].unique())

    # Classify each source as Supplier (all top5 targets are own planets) or Conqueror
    top5_with_mine = top5_ids.copy()
    top5_with_mine["target_is_mine"] = top5_with_mine["id"].isin(mine_src_ids)
    src_nature = (
        top5_with_mine
        .groupby("id_src")
        .agg(mine_count=("target_is_mine", "sum"), total_count=("target_is_mine", "count"))
        .reset_index()
    )
    src_nature["status"] = np.where(
        src_nature["mine_count"] == src_nature["total_count"], "Supplier", "Conqueror"
    )
    conqueror_ids = set(src_nature.loc[src_nature["status"] == "Conqueror", "id_src"])
    supplier_ids  = set(src_nature.loc[src_nature["status"] == "Supplier",  "id_src"])

    # ── Conqueror: attack enemy/neutral planets ──────────────────────────────
    attacks_conqueror = pd.DataFrame()
    conqueror_needs = None
    if conqueror_ids:
        _c = (
            attacks_with_angle[attacks_with_angle["id_src"].isin(conqueror_ids)]
            .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            .query("is_top5")
            .loc[lambda d: d["owner"] != player_id]
            .assign(ships_needed=lambda d: np.where(
                d["owner"] == -1, d["ships"], d["ships"] + d["production"]
            ))
            .loc[lambda d:
                (d["ships_needed"] + 1 <= d["ships_sent"]) &
                (d["ships_sent"] <= d["ships_needed"] + d["production_src"] + 1)
            ]
            .sort_values(["step", "ships_sent"])
            .groupby(["id_src", "id"], sort=False).first().reset_index()
            .assign(time_cost=lambda d: d["ships_needed"] / d["production_src"])
        )
        if not _c.empty:
            conqueror_needs = (
                _c
                .groupby("id_src", sort=False)
                .agg(
                    ship_min=("ships_min", "min"),
                    all_need=("ships_sent", "sum"),
                    lowest_need=("ships_sent", "min"),
                    nb_need=("ships_sent", "count"),
                )
            )
            attacks_conqueror = (
                _c
                .assign(
                    total_time_cost=_c.groupby("id_src")["time_cost"].transform("sum")
                ).assign(
                    score=lambda d: (
                        (d["total_time_cost"] - d["time_cost"] - d["step_diff"]) * d["production"]
                    )
                )
                # .loc[lambda d: d["score"] > 0]
                .sort_values("score", ascending=False)
                .groupby("id_src", sort=False).first().reset_index()
                .loc[lambda d: d["ships_sent"] <= d["ships_min"]]
            )

    # ── Supplier: reinforce own planets ─────────────────────────────────────
    attacks_supplier = pd.DataFrame()
    if supplier_ids and conqueror_needs is not None:
        _s = (
            attacks_with_angle[attacks_with_angle["id_src"].isin(supplier_ids)]
            .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            .loc[lambda d: d["is_top5"]]
            .assign(target_is_supplier=lambda d: d["id"].isin(supplier_ids))
            .query("not target_is_supplier")
            .merge(
                conqueror_needs,
                left_on="id",
                right_on="id_src",
                how="right"
            )
            .query("(lowest_need - ships_min) * 1.5 < ships_sent")
            .query("ships_min * 0.75 < ships_sent < ships_min")
            .sort_values(["all_need", "ships_sent"], ascending=[False, True])
            .groupby(["id_src"], sort=False).first().reset_index()
        )
        attacks_supplier = _s

    # ── Combine and emit ─────────────────────────────────────────────────────
    parts = [df for df in [attacks_conqueror, attacks_supplier] if not df.empty]
    if not parts:
        return moves

    attacks = pd.concat(parts, ignore_index=True)
    print("Currently using testing _04_score_and_decide")
    for _, row in attacks.iterrows():
        print(f"From {row['id_src']}, To {row['id']} at step {row['step']} "
              f"with {row['ships_sent']} ships (target has min {row['ships_min']})")

    moves += attacks[["id_src", "final_angle", "ships_sent"]].values.tolist()
    return moves

## Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy

Planet 1 (x=30) should attack planet 2 (x=70). Planet 0 (x=20) stays as supplier.

In [4]:
obs01 = Obs(
    planets=[
        [0, 0, 5.0, 5.0, 1 + math.log(3), 50, 3],  # Supplier  (dist≈56.6, not orbiting)
        [1, 0, 15.0,  5.0, 1 + math.log(3), 50, 3],  # Conqueror (dist≈51.5, not orbiting)
        [2, 1, 25.0,  5.0, 1.0,              1,  1],  # Enemy     (dist≈51.5, not orbiting)
    ],
    angular_velocity=0.05,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.0,5.0,2.098612,50,3,0,fix
1,0,1,15.0,5.0,2.098612,50,3,0,fix
2,0,2,25.0,5.0,1.000000,1,1,1,fix
3,1,0,5.0,5.0,2.098612,53,3,0,fix
4,1,1,15.0,5.0,2.098612,53,3,0,fix
5,1,2,25.0,5.0,1.000000,2,1,1,fix
6,2,0,5.0,5.0,2.098612,56,3,0,fix
7,2,1,15.0,5.0,2.098612,56,3,0,fix
8,2,2,25.0,5.0,1.000000,3,1,1,fix
9,3,0,5.0,5.0,2.098612,59,3,0,fix


In [5]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,10.0,10.0,7.901388,8.524700,0.000000,0.161830,6.121355,0.161830,0.000000
1,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,10.0,10.0,7.901388,8.556020,0.000000,0.164822,6.118363,0.164822,0.000000
2,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,10.0,10.0,7.901388,8.586829,0.000000,0.167626,6.115559,0.167626,0.000000
3,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,10.0,10.0,7.901388,8.617142,0.000000,0.170258,6.112928,0.170258,0.000000
4,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,10.0,10.0,7.901388,9.069999,0.000000,0.197862,6.085323,0.197862,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,20.0,20.0,19.858866,21.000000,0.049680,0.000000,6.233505,0.049680,0.000000
684,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,20.0,20.0,19.271312,21.000000,0.034885,0.000000,6.248300,0.034885,0.000000
685,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,20.0,20.0,20.907561,21.000000,0.020536,0.000000,6.262650,0.020536,0.000000
686,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,0.000000,20.0,20.0,20.402100,21.000000,0.045330,0.000000,6.237855,0.045330,0.000000


In [6]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,7.901388,8.524700,0.000000,0.161830,6.121355,0.161830,0.000000,0.000000
1,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,7.901388,8.556020,0.000000,0.164822,6.118363,0.164822,0.000000,0.000000
2,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,7.901388,8.586829,0.000000,0.167626,6.115559,0.167626,0.000000,0.000000
3,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,7.901388,8.617142,0.000000,0.170258,6.112928,0.170258,0.000000,0.000000
4,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,7.901388,9.069999,0.000000,0.197862,6.085323,0.197862,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558,1,0,15.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,10.198612,11.000000,0.097087,0.000000,6.186098,0.097087,0.000000,0.000000
559,1,0,15.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,10.198612,11.198612,0.207246,0.162965,2.934347,3.348838,3.141593,3.141593
560,1,0,15.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,11.470043,12.098612,0.139959,0.000000,3.001634,3.281552,3.141593,3.141593
561,0,0,5.0,5.0,2.098612,50,3,fix,0,11,...,10.0,10.0,11.198612,12.098612,0.162965,0.000000,6.120220,0.162965,0.000000,0.000000


In [7]:
action01 = _04_score_and_decide(safe01, player_id=0)
action01

Currently using testing _04_score_and_decide
From 1, To 2 at step 4 with 7 ships (target has min 50)
From 0, To 1 at step 2 with 38 ships (target has min 50)


[[1.0, 0.0, 7.0], [0.0, 0.0, 38.0]]

In [8]:

print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 20)
make_animation(snaps01, title='Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy', interval=200)

Action: [[1.0, 0.0, 7.0], [0.0, 0.0, 38.0]]


## Test 02 — 1 Supplier, 2 Conquerors (one more in need)

Planet 1 attacks planet 2 (easy). Planet 3 attacks planet 4 (heavy — 100 ships). Planet 0 stays as supplier.

In [9]:
obs02 = Obs(
    planets=[
        [0, 0, 5.0, 5.0, 1 + math.log(3), 150,  3],  # dist≈56.6, not orbiting
        [1, 0, 15.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [2, 1, 25.0,  5.0, 1 + math.log(3), 1,   1],  # dist≈51.5, not orbiting
        [3, 0,  5.0, 15.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [4, 1,  5.0, 25.0, 1 + math.log(3), 100, 1],  # dist≈51.5, not orbiting
    ],
    angular_velocity=0.05,
)
df_s02, pd02 = StrategyPipeline._01_get_obs_dataframe(obs02, step=0, num_agents=2)
df_s02

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.0,5.0,2.098612,150,3,0,fix
1,0,1,15.0,5.0,2.098612,50,3,0,fix
2,0,2,25.0,5.0,2.098612,1,1,1,fix
3,0,3,5.0,15.0,2.098612,50,3,0,fix
4,0,4,5.0,25.0,2.098612,100,1,1,fix
5,1,0,5.0,5.0,2.098612,153,3,0,fix
6,1,1,15.0,5.0,2.098612,53,3,0,fix
7,1,2,25.0,5.0,2.098612,2,1,1,fix
8,1,3,5.0,15.0,2.098612,53,3,0,fix
9,1,4,5.0,25.0,2.098612,101,1,1,fix


In [10]:
pa02 = StrategyPipeline._02_get_all_opportunities(df_s02, pd02, player_id=0)
pa02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,10.0,10.0,7.901388,8.289440,0.000000e+00,0.133635,6.149550,0.133635,0.0
1,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,10.0,10.0,7.901388,8.253267,0.000000e+00,0.128129,6.155056,0.128129,0.0
2,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,10.0,10.0,7.901388,8.216375,0.000000e+00,0.122072,6.161113,0.122072,0.0
3,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,10.0,10.0,7.901388,8.178730,0.000000e+00,0.115357,6.167829,0.115357,0.0
4,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,10.0,10.0,7.901388,8.140301,0.000000e+00,0.107836,6.175349,0.107836,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3211,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,20.0,20.0,19.271312,21.168278,1.002872e-01,0.084754,6.182898,0.100287,0.0
3212,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,20.0,20.0,18.630980,20.456798,8.242267e-02,0.101308,6.181877,0.101308,0.0
3213,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,20.0,20.0,17.926694,19.674259,1.716028e-02,0.104561,6.178624,0.104561,0.0
3214,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,0.0,20.0,20.0,17.901388,18.803785,1.490116e-08,0.088945,6.194240,0.088945,0.0


In [11]:
safe02 = StrategyPipeline._03_filter_collision(pa02)
safe02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,10.000000,10.000000,7.901388,8.289440,0.000000e+00,0.133635,6.149550,0.133635,0.000000,0.000000
1,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,10.000000,10.000000,7.901388,8.253267,0.000000e+00,0.128129,6.155056,0.128129,0.000000,0.000000
2,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,10.000000,10.000000,7.901388,8.216375,0.000000e+00,0.122072,6.161113,0.122072,0.000000,0.000000
3,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,10.000000,10.000000,7.901388,8.178730,0.000000e+00,0.115357,6.167829,0.115357,0.000000,0.000000
4,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,10.000000,10.000000,7.901388,8.140301,0.000000e+00,0.107836,6.175349,0.107836,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2435,3,0,5.0,15.0,2.098612,50,3,fix,0,11,...,14.142136,14.142136,12.628972,13.787901,1.088624e-01,0.148268,5.349519,5.646055,-0.785398,-0.785398
2436,3,0,5.0,15.0,2.098612,50,3,fix,0,11,...,14.142136,14.142136,12.043523,12.198612,2.107342e-08,0.060291,5.437497,5.558078,-0.785398,-0.785398
2437,3,0,5.0,15.0,2.098612,50,3,fix,0,11,...,10.000000,10.000000,11.198612,12.098612,1.629649e-01,0.000000,4.549424,4.875354,-1.570796,-1.570796
2438,0,0,5.0,5.0,2.098612,150,3,fix,0,11,...,10.000000,10.000000,11.198612,12.098612,1.629649e-01,0.000000,1.407831,1.733761,1.570796,1.570796


In [16]:
action02 = _04_score_and_decide(safe02, player_id=0)
print("Action:", action02)
snaps02 = simulate_with_action(copy.deepcopy(obs02), action02, 20)
make_animation(snaps02, title='Test 02 — 1 Supplier, 2 Conquerors (one more in need)', interval=200)

Currently using testing _04_score_and_decide
From 1, To 2 at step 4 with 7 ships (target has min 50)
From 3, To 2 at step 9 with 12 ships (target has min 50)
From 0, To 3 at step 2 with 113 ships (target has min 150)
Action: [[1.0, 0.0, 7.0], [3.0, -0.4636476090008061, 12.0], [0.0, 1.5707963267948966, 113.0]]


## Test 03 — 4 Suppliers, 2 Conquerors (one more in need)

Same enemies as Test 02. Planets 5, 6, 7 added as extra suppliers. Pipeline should still route correctly.

In [17]:
obs03 = Obs(
    planets=[
        [0, 0, 8.0, 8.0, 1 + math.log(3), 50,  3],  # dist≈56.6, not orbiting
        [1, 0, 20.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [2, 1, 30.0,  5.0, 1 + math.log(3), 1,   1],  # dist≈51.5, not orbiting
        [3, 0,  5.0, 20.0, 1 + math.log(3), 10,  3],  # dist≈51.5, not orbiting
        [4, 1,  5.0, 30.0, 1 + math.log(3), 20, 1],  # dist≈51.5, not orbiting
        [5, 0,  2.0,  2.0, 1 + math.log(3), 50,  3],  # dist≈63.6, not orbiting
        [6, 0,  5.0, 15.0, 1 + math.log(3), 50,  3],  # dist≈57.0, not orbiting
        [7, 0, 15.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈57.0, not orbiting
        [8, 0, 12.0, 12.0, 1 + math.log(3), 50,  3],  # dist≈56.6, not orbiting
    ],
    angular_velocity=0.05,
)
df_s03, pd03 = StrategyPipeline._01_get_obs_dataframe(obs03, step=0, num_agents=2)
df_s03

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,8.0,8.0,2.098612,50,3,0,fix
1,0,1,20.0,5.0,2.098612,50,3,0,fix
2,0,2,30.0,5.0,2.098612,1,1,1,fix
3,0,3,5.0,20.0,2.098612,10,3,0,fix
4,0,4,5.0,30.0,2.098612,20,1,1,fix
...,...,...,...,...,...,...,...,...,...
94,10,4,5.0,30.0,2.098612,30,1,1,fix
95,10,5,2.0,2.0,2.098612,80,3,0,fix
96,10,6,5.0,15.0,2.098612,80,3,0,fix
97,10,7,15.0,5.0,2.098612,80,3,0,fix


In [18]:
pa03 = StrategyPipeline._02_get_all_opportunities(df_s03, pd03, player_id=0)
pa03

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,-0.404892,7.615773,7.615773,5.517161,5.573078,0.000000,0.073882,5.804412,5.952175,-0.404892
1,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,-0.404892,7.615773,7.615773,5.517161,5.560352,0.000000,0.065103,5.813191,5.943396,-0.404892
2,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,-0.404892,7.615773,7.615773,5.517161,5.547454,0.000000,0.054667,5.823626,5.932961,-0.404892
3,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,-0.404892,7.615773,7.615773,5.517161,5.534380,0.000000,0.041327,5.836966,5.919621,-0.404892
4,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,-0.404892,7.615773,7.615773,5.517161,5.521125,0.000000,0.019884,5.858409,5.898178,-0.404892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9972,6,0,5.0,15.0,2.098612,50,3,fix,0,11,...,-0.380506,26.925824,26.925824,28.759594,29.024436,0.036675,0.000000,5.866004,5.939354,-0.380506
9973,6,0,5.0,15.0,2.098612,50,3,fix,0,11,...,-0.380506,26.925824,26.925824,28.936213,29.024436,0.021571,0.000000,5.881108,5.924250,-0.380506
9974,8,0,12.0,12.0,2.098612,50,3,fix,0,11,...,1.941688,19.313208,19.313208,19.858866,21.411820,0.103519,0.000000,1.838168,2.045207,1.941688
9975,8,0,12.0,12.0,2.098612,50,3,fix,0,11,...,1.941688,19.313208,19.313208,19.271312,21.168278,0.108812,0.048536,1.832876,2.050500,1.941688


In [19]:
safe03 = StrategyPipeline._03_filter_collision(pa03)
safe03

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,7.615773,7.615773,5.517161,5.573078,0.000000,0.073882,5.804412,5.952175,-0.404892,-0.404892
1,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,7.615773,7.615773,5.517161,5.560352,0.000000,0.065103,5.813191,5.943396,-0.404892,-0.404892
2,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,7.615773,7.615773,5.517161,5.547454,0.000000,0.054667,5.823626,5.932961,-0.404892,-0.404892
3,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,7.615773,7.615773,5.517161,5.534380,0.000000,0.041327,5.836966,5.919621,-0.404892,-0.404892
4,0,0,8.0,8.0,2.098612,50,3,fix,0,11,...,7.615773,7.615773,5.517161,5.521125,0.000000,0.019884,5.858409,5.898178,-0.404892,-0.404892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6056,8,0,12.0,12.0,2.098612,50,3,fix,0,11,...,19.313208,19.313208,20.402100,21.411820,0.090408,0.000000,1.851279,2.032096,1.941688,1.941688
6057,8,0,12.0,12.0,2.098612,50,3,fix,0,11,...,19.313208,19.313208,19.858866,21.411820,0.103519,0.000000,5.808775,6.015813,-0.370891,-0.370891
6058,8,0,12.0,12.0,2.098612,50,3,fix,0,11,...,19.313208,19.313208,21.380429,21.411820,0.017796,0.000000,5.894498,5.930090,-0.370891,-0.370891
6059,8,0,12.0,12.0,2.098612,50,3,fix,0,11,...,19.313208,19.313208,19.858866,21.411820,0.103519,0.000000,1.838168,2.045207,1.941688,1.941688


In [20]:
action03 = _04_score_and_decide(safe03, player_id=0)
print("Action:", action03)
snaps03 = simulate_with_action(copy.deepcopy(obs03), action03, 20)
make_animation(snaps03, title='Test 03 — 4 Suppliers, 2 Conquerors (one more in need)', interval=200)

C:\Users\trant\AppData\Local\Temp\ipykernel_6480\3954862869.py:110: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .assign(is_top5=lambda d: d["is_top5"].fillna(False))


Currently using testing _04_score_and_decide
From 1, To 2 at step 4 with 7 ships (target has min 50)
From 7, To 1 at step 1 with 38 ships (target has min 50)
From 8, To 1 at step 3 with 38 ships (target has min 50)
From 6, To 3 at step 1 with 38 ships (target has min 50)
Action: [[1.0, 0.0, 7.0], [7.0, 0.0, 38.0], [8.0, -0.7188299996216245, 38.0], [6.0, 1.5707963267948966, 38.0]]


## Test 04 — 4 Suppliers, 1 Conqueror Orbiting

Planets 4 and 5 orbit (dist_from_center < 50 − radius). `angular_velocity=0.05`. The pipeline must intercept the moving enemy planet.

In [39]:
obs04 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50, 3],  # dist≈56.6, not orbiting
        [1, 0,  5.0,  5.0, 1 + math.log(3), 100, 3],  # dist≈63.6, not orbiting
        [2, 0,  5.0, 15.0, 1 + math.log(3), 50, 3],  # dist≈57.0, not orbiting
        [3, 0, 15.0,  5.0, 1 + math.log(3), 50, 3],  # dist≈57.0, not orbiting
        [4, 0, 20.0, 20.0, 1.0,              50, 1],  # dist≈42.4 < 49, orbiting
        [5, 1, 25.0, 25.0, 1.0,              50, 1],  # dist≈35.4 < 49, orbiting enemy
    ],
    angular_velocity=0.05,
)
df_s04, pd04 = StrategyPipeline._01_get_obs_dataframe(obs04, step=0, num_agents=2)
df_s04

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,10.000000,10.000000,2.098612,50,3,0,fix
1,0,1,5.000000,5.000000,2.098612,100,3,0,fix
2,0,2,5.000000,15.000000,2.098612,50,3,0,fix
3,0,3,15.000000,5.000000,2.098612,50,3,0,fix
4,0,4,20.000000,20.000000,1.000000,50,1,0,moving
...,...,...,...,...,...,...,...,...,...
61,10,1,5.000000,5.000000,2.098612,130,3,0,fix
62,10,2,5.000000,15.000000,2.098612,80,3,0,fix
63,10,3,15.000000,5.000000,2.098612,80,3,0,fix
64,10,4,36.035553,9.937621,1.000000,60,1,0,moving


In [40]:
pa04 = StrategyPipeline._02_get_all_opportunities(df_s04, pd04, player_id=0)
pa04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.361656,0.000000,0.198041,3.728950,4.125032,-2.356194
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.345731,0.000000,0.194630,3.732361,4.121621,-2.356194
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.329530,0.000000,0.191041,3.735950,4.118032,-2.356194
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.313044,0.000000,0.187258,3.739733,4.114249,-2.356194
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.296263,0.000000,0.183261,3.743729,4.110252,-2.356194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4376,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.002396,24.060491,26.035628,24.365583,26.828580,0.039334,0.023054,6.257735,0.067834,0.013052
4377,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,0.006520,24.060491,25.430615,24.668518,26.403562,0.032589,0.008916,6.279096,0.061088,0.017510
4378,4,0,20.0,20.0,1.000000,50,1,moving,0,11,...,-1.892547,15.811388,15.811388,17.532367,17.910001,0.072149,0.000000,4.318489,4.462788,-1.892547
4379,1,0,5.0,5.0,2.098612,100,3,fix,0,11,...,0.157773,30.004380,31.425875,29.416320,31.710814,0.027226,0.030366,0.127408,0.212225,0.171386


In [41]:
safe04 = StrategyPipeline._03_filter_collision(pa04)
safe04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.361656,0.000000,0.198041,3.728950,4.125032,-2.356194,-2.356194
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.345731,0.000000,0.194630,3.732361,4.121621,-2.356194,-2.356194
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.329530,0.000000,0.191041,3.735950,4.118032,-2.356194,-2.356194
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.313044,0.000000,0.187258,3.739733,4.114249,-2.356194,-2.356194
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.296263,0.000000,0.183261,3.743729,4.110252,-2.356194,-2.356194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3734,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,24.060491,26.035628,23.721686,26.113139,0.039385,0.038239,6.242551,0.067884,0.013052,0.013052
3735,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,24.060491,26.035628,24.050342,26.478312,0.041572,0.034153,6.246636,0.070071,0.013052,0.013052
3736,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,24.060491,26.035628,24.365583,26.828580,0.039334,0.023054,6.257735,0.067834,0.013052,0.013052
3737,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,24.060491,25.430615,24.668518,26.403562,0.032589,0.008916,6.279096,0.061088,0.017510,0.017510


In [45]:
action04 = _04_score_and_decide(safe04, player_id=0)
print("Action:", action04)
snaps04 = simulate_with_action(copy.deepcopy(obs04), action04, 20)
make_animation(snaps04, title='Test 04 — 4 Suppliers, 1 Conqueror Orbiting', interval=200)

Currently using testing _04_score_and_decide
From 1, To 5 at step 10 with 62 ships (target has min 100)
Action: [[1.0, 0.34199438971699275, 62.0]]


## Test 05 — Our Agent vs Random Agent

Full game via `kaggle_environments`. Player 0 uses `agent()` from `82-...py`, player 1 uses a random policy.

In [25]:
def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]


# Reset agent globals so this cell is re-runnable
step = 0
num_agents = None
player_id = None

SEED = 42
N_STEPS = 100
random.seed(SEED)

env = ke.make("orbit_wars", debug=False)
env.reset(2)

snaps05 = []
for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation
    snaps05.append({
        'step':    env_step,
        'planets': [list(p) for p in obs0.planets],
        'fleets':  [list(f) for f in obs0.fleets],
    })
    action0 = agent(obs0)
    action1 = random_agent_fn(obs1)
    env.step([action0, action1])
    if env.state[0].status != "ACTIVE":
        break

obs0 = env.state[0].observation
snaps05.append({
    'step':    len(snaps05),
    'planets': [list(p) for p in obs0.planets],
    'fleets':  [list(f) for f in obs0.fleets],
})
p0 = sum(p[5] for p in obs0.planets if p[1] == 0)
p1 = sum(p[5] for p in obs0.planets if p[1] == 1)
winner = "Our agent wins" if p0 > p1 else "Random wins" if p1 > p0 else "Tie"
print(f"After {len(snaps05) - 1} steps: {winner}  (player0={p0}, player1={p1})")

make_animation(snaps05, title='Test 05 — Our Agent vs Random', interval=100)

NameError: name 'ke' is not defined